In [1]:
def compute_weighted_means(data_subset):
    """Return a dict of weighted mean vectors for each target."""
    weights = np.array([sample.get("signal_to_noise", 1.0) for sample in data_subset])
    mean_dict = {}
    for tgt in targets:
        arr_list = []
        w_list = []
        for sample, w in zip(data_subset, weights):
            if tgt in sample and sample[tgt] is not None:
                arr_list.append(sample[tgt])
                w_list.append(w)
        if not arr_list:
            mean_dict[tgt] = np.zeros(0)
            continue
        arrs = np.vstack(arr_list)  # (n_samples, seq_scored)
        w_arr = np.array(w_list)
        mean_vec = np.average(arrs, axis=0, weights=w_arr)
        mean_dict[tgt] = mean_vec
    return mean_dict


base_mean_vectors = compute_weighted_means(train_subset)

target_bounds = {}
for tgt in targets:
    vals = []
    for sample in train_subset:
        seq_scored = sample["seq_scored"]
        vals.extend(sample[tgt][:seq_scored])
    target_bounds[tgt] = (float(np.min(vals)), float(np.max(vals)))

max_scored = max(sample["seq_scored"] for sample in val_subset)  # typically 68
position_offsets = {}
for tgt in targets:
    pos_vals = [[] for _ in range(max_scored)]
    for sample in val_subset:
        seq_scored = sample["seq_scored"]
        true_arr = np.array(sample[tgt][:seq_scored])
        for pos in range(seq_scored):
            pos_vals[pos].append(true_arr[pos])
    offsets = np.empty(max_scored, dtype=float)
    for pos in range(max_scored):
        if pos_vals[pos]:
            offsets[pos] = np.mean(pos_vals[pos]) - base_mean_vectors[tgt][pos]
        else:
            offsets[pos] = 0.0
    position_offsets[tgt] = offsets

val_weights = np.array([sample.get("signal_to_noise", 1.0) for sample in val_subset])
scaling_factors_pos = {}
for tgt in targets:
    sf_raw = np.ones(max_scored, dtype=float)
    for pos in range(max_scored):
        num = 0.0
        den = 0.0
        for sample, w in zip(val_subset, val_weights):
            if pos < sample["seq_scored"]:
                true_val = sample[tgt][pos]
                base_val = base_mean_vectors[tgt][pos]
                num += w * true_val
                den += w * base_val
        sf_raw[pos] = (num / den) if den != 0 else 1.0
    # use a milder smoothing toward 1.0 (0.5 * raw + 0.5 * 1.0) and keep clipping
    sf_smoothed = 0.5 * sf_raw + 0.5 * 1.0
    sf_clipped = np.clip(sf_smoothed, 0.8, 1.2)
    scaling_factors_pos[tgt] = sf_clipped


def weighted_linear_fit(x, y, w):
    """Weighted least‑squares fit of y = a*x + b."""
    w_sum = np.sum(w)
    if w_sum == 0:
        return 1.0, 0.0
    x_mean = np.sum(w * x) / w_sum
    y_mean = np.sum(w * y) / w_sum
    cov = np.sum(w * (x - x_mean) * (y - y_mean))
    var = np.sum(w * (x - x_mean) ** 2)
    a = cov / var if var != 0 else 1.0
    b = y_mean - a * x_mean
    return a, b


calibration_factors = {}  # tgt -> (slope, intercept)
for tgt in targets:
    xs, ys, ws = [], [], []
    for sample, w in zip(val_subset, val_weights):
        seq_scored = sample["seq_scored"]
        for pos in range(seq_scored):
            base = base_mean_vectors[tgt][pos]
            scaled = scaling_factors_pos[tgt][pos] * base + position_offsets[tgt][pos]
            xs.append(scaled)
            ys.append(sample[tgt][pos])
            ws.append(w)
    xs = np.array(xs)
    ys = np.array(ys)
    ws = np.array(ws)
    slope, intercept = weighted_linear_fit(xs, ys, ws)
    calibration_factors[tgt] = (slope, intercept)




NameError: name 'train_subset' is not defined

In [2]:
test_path = next(Path(".").rglob("test.json"))
test_data = []
with open(test_path, "r") as f:
    for line in f:
        line = line.strip()
        if line:
            test_data.append(json.loads(line))

shrink_factor = (
    1.0  # use the full calibrated prediction without blending back to the raw base mean
)

prediction_rows = []
for sample in test_data:
    sample_id = sample["id"]
    seq_len = sample["seq_length"]
    seq_scored = sample["seq_scored"]
    for pos in range(seq_len):
        row = {"id_seqpos": f"{sample_id}_{pos}"}
        for tgt in targets:
            if pos < seq_scored:
                base = base_mean_vectors[tgt][pos]
                pred_base = (
                    scaling_factors_pos[tgt][pos] * base + position_offsets[tgt][pos]
                )
                slope, intercept = calibration_factors[tgt]
                calibrated = slope * pred_base + intercept
                pred = shrink_factor * calibrated + (1 - shrink_factor) * base
                lo, hi = target_bounds[tgt]
                pred = float(np.clip(pred, lo, hi))
                row[tgt] = pred
            else:
                row[tgt] = 0.0
        prediction_rows.append(row)

pred_df = pd.DataFrame(prediction_rows)

expected_cols = [
    "id_seqpos",
    "reactivity",
    "deg_Mg_pH10",
    "deg_pH10",
    "deg_Mg_50C",
    "deg_50C",
]
pred_df = pred_df[expected_cols]




NameError: name 'Path' is not defined

In [3]:
pred_df.to_csv("submission.csv", index=False)
print(
    "Submission file 'submission.csv' written successfully with", len(pred_df), "rows."
)

NameError: name 'pred_df' is not defined